# Yapay Zeka ve Üretken Modeller Atölyesi - Hafta 5
## Görüntü İşleme, Transfer Learning ve Önceden Eğitilmiş CNN Mimarileri

**BTK Akademi** | Dr.Öğr.Üyesi Furkan Göz

---

### Bu Haftanın İçeriği

1. **Görüntü İşleme ve Derin Öğrenme İlişkisi**
2. **Temel Görüntü İşleme Teknikleri** (Grayscale, Blur, Thresholding, Canny, Histogram Eşitleme)
3. **Morfolojik ve Geometrik İşlemler**
4. **Transfer Learning Nedir ve Neden Kullanılır?**
5. **Önceden Eğitilmiş Modeller** (VGG-16, ResNet50, EfficientNet, MobileNetV2, DenseNet, ConvNeXt)
6. **Transfer Learning Uygulama** (Feature Extraction + Fine Tuning)
7. **Model Değerlendirme** (Confusion Matrix, Grad-CAM)
8. **Video İşleme ve Gerçek Zamanlı Çıkarım**

---
## 1. Görüntü İşleme ve Derin Öğrenme İlişkisi

Evrişimsel sinir ağları (CNN), ham piksel matrisleri üzerinden otomatik özellik çıkarımı yapar. Ancak bu ağlara verilen girdinin kalitesi, modelin başarısını doğrudan etkiler.

**Görüntü işleme algoritmaları** ham veriyi modelin daha kolay işleyebileceği formata dönüştürür:
- Piksel matrislerindeki gürültüleri temizler
- Hedef nesneleri belirginleştirir
- Veri setinin yapısal kalitesini artırır
- Ağın optimizasyon sürecini hızlandırır
- YOLO gibi nesne tespiti modellerine nitelikli girdi sağlar

---
## 2. Temel Görüntü İşleme Teknikleri

Aşağıda OpenCV kütüphanesi kullanılarak temel görüntü işleme tekniklerinin uygulamaları yer almaktadır.

In [ ]:
# Gerekli kütüphanelerin kurulumu
# !pip install opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Matplotlib'in notebook içinde görsel göstermesi için
%matplotlib inline

# Türkçe karakter desteği için
plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
# Örnek bir görsel yükleme (kendi görselinizin yolunu yazın)
# img = cv2.imread('ornek_gorsel.jpg')

# Demo amaçlı rastgele bir görsel oluşturalım
# Gerçek uygulamada kendi görselinizi yükleyin
np.random.seed(42)
img = np.random.randint(50, 200, (300, 400, 3), dtype=np.uint8)

# Daha gerçekçi bir demo için gradient görsel oluşturalım
img_demo = np.zeros((300, 400, 3), dtype=np.uint8)
for i in range(300):
    for j in range(400):
        img_demo[i, j] = [int(i * 0.85), int(j * 0.64), int((i + j) * 0.36)]

# Gürültü ekleyelim
noise = np.random.normal(0, 25, img_demo.shape).astype(np.int16)
img_noisy = np.clip(img_demo.astype(np.int16) + noise, 0, 255).astype(np.uint8)

print(f"Görsel boyutu: {img_noisy.shape}")
print(f"Veri tipi: {img_noisy.dtype}")
print(f"Piksel değer aralığı: [{img_noisy.min()}, {img_noisy.max()}]")

### 2.1 Grayscale (Gri Tonlama)

Görüntü matrisini üç kanallı RGB yapıdan tek kanallı yapıya indirger. Renk bilgisinin önemsiz olduğu yapısal analizlerde tercih edilir.

**Matematiksel formül:** `Gray = 0.299*R + 0.587*G + 0.114*B`

Bu ağırlıklar insan gözünün farklı renklere olan hassasiyetine göre belirlenmiştir.

In [ ]:
# Grayscale dönüşümü
gray = cv2.cvtColor(img_noisy, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img_noisy, cv2.COLOR_BGR2RGB))
axes[0].set_title('Orijinal (3 Kanal - RGB)')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale (1 Kanal)')
axes[1].axis('off')

plt.suptitle('Grayscale Dönüşümü', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Orijinal shape: {img_noisy.shape} -> Grayscale shape: {gray.shape}")

### 2.2 Gaussian ve Median Blur (Bulanıklaştırma)

Görüntü matrisi üzerindeki istenmeyen hatalı pikselleri (gürültü) gidermek amacıyla uygulanan filtrelerdir.

- **Gaussian Blur:** Her pikseli, komşu piksellerin Gaussian dağılımına göre ağırlıklı ortalamasıyla değiştirir.
- **Median Blur:** Her pikseli, komşu piksellerin medyan değeriyle değiştirir. Tuz-biber gürültüsüne karşı daha etkilidir.

In [ ]:
# Gaussian Blur
# kernel_size: Filtre boyutu (tek sayı olmalı). Büyüdükçe daha fazla bulanıklaşma olur.
gaussian_blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Median Blur
median_blur = cv2.medianBlur(gray, 5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Orijinal (Gürültülü)')

axes[1].imshow(gaussian_blur, cmap='gray')
axes[1].set_title('Gaussian Blur (5x5)')

axes[2].imshow(median_blur, cmap='gray')
axes[2].set_title('Median Blur (5x5)')

for ax in axes:
    ax.axis('off')

plt.suptitle('Blur Filtreleri Karşılaştırması', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Thresholding (Eşikleme)

Piksel değerlerini belirlenen eşik değerine göre siyah ve beyaz renklere ayırır. 

- **Basit Eşikleme:** Sabit bir eşik değeri kullanılır.
- **Otsu Eşikleme:** Eşik değeri otomatik olarak histogram analizi ile hesaplanır (bimodal dağılım varsayılır).

In [ ]:
# Basit Thresholding
# 127 eşik değeri: Bu değerin altı -> 0 (siyah), üstü -> 255 (beyaz)
_, thresh_simple = cv2.threshold(gaussian_blur, 127, 255, cv2.THRESH_BINARY)

# Otsu Thresholding (Otomatik eşik hesaplama)
# Otsu algoritması piksellerin histogram dağılımını analiz ederek
# sınıflar arası varyansı maksimize eden eşik değerini bulur
otsu_thresh_val, thresh_otsu = cv2.threshold(gaussian_blur, 0, 255, 
                                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gaussian_blur, cmap='gray')
axes[0].set_title('Bulanıklaştırılmış Görsel')

axes[1].imshow(thresh_simple, cmap='gray')
axes[1].set_title('Basit Eşikleme (threshold=127)')

axes[2].imshow(thresh_otsu, cmap='gray')
axes[2].set_title(f'Otsu Eşikleme (threshold={otsu_thresh_val:.0f})')

for ax in axes:
    ax.axis('off')

plt.suptitle('Thresholding (Eşikleme) Teknikleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Otsu algoritmasının hesapladığı optimal eşik değeri: {otsu_thresh_val}")

### 2.4 Canny Kenar Tespiti

Görüntü içindeki nesnelerin ayırt edici yapısal sınırlarını matematiksel olarak belirler.

**Çalışma Adımları:**
1. Gaussian Blur ile gürültü azaltma
2. Sobel operatörü ile gradyan hesaplama
3. Non-maximum suppression (gradyan yönünde olmayan pikselleri eleme)
4. Çift eşikleme ile kenar belirleme (hysteresis thresholding)

In [ ]:
# Canny Kenar Tespiti
# threshold1: Alt eşik - Bu değerin altındaki gradyanlar kenar olarak kabul edilmez
# threshold2: Üst eşik - Bu değerin üstündeki gradyanlar kesinlikle kenardır
edges_canny = cv2.Canny(gaussian_blur, 50, 150)

# Sobel Kenar Tespiti (X ve Y yönünde)
sobel_x = cv2.Sobel(gaussian_blur, cv2.CV_64F, 1, 0, ksize=3)  # Yatay kenarlar
sobel_y = cv2.Sobel(gaussian_blur, cv2.CV_64F, 0, 1, ksize=3)  # Dikey kenarlar
sobel_combined = cv2.magnitude(sobel_x, sobel_y)

# Laplacian Kenar Tespiti
laplacian = cv2.Laplacian(gaussian_blur, cv2.CV_64F)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(gaussian_blur, cmap='gray')
axes[0, 0].set_title('Orijinal')

axes[0, 1].imshow(edges_canny, cmap='gray')
axes[0, 1].set_title('Canny Kenar Tespiti')

axes[1, 0].imshow(sobel_combined, cmap='gray')
axes[1, 0].set_title('Sobel (X + Y Birleşik)')

axes[1, 1].imshow(np.abs(laplacian), cmap='gray')
axes[1, 1].set_title('Laplacian')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Kenar Tespiti Yöntemleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Kontur Tespiti

Tespit edilen kenar piksellerini birbirine bağlayarak kapalı şekil formatlarına dönüştürür.

In [ ]:
# Kontur tespiti için önce eşikleme yapılır
contours, hierarchy = cv2.findContours(thresh_otsu, cv2.RETR_TREE, 
                                        cv2.CHAIN_APPROX_SIMPLE)

# Konturları çiz
img_contours = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_contours, contours, -1, (0, 255, 0), 2)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img_contours, cv2.COLOR_BGR2RGB))
plt.title(f'Kontur Tespiti ({len(contours)} kontur bulundu)')
plt.axis('off')
plt.show()

### 2.6 Histogram Eşitleme

Matris üzerindeki ışık dağılımını dengeler ve düşük ışıklı görsellerin genel kontrast oranını artırır.

In [ ]:
# Histogram Eşitleme
equalized = cv2.equalizeHist(gray)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Orijinal Görsel')
axes[0, 0].axis('off')

axes[0, 1].hist(gray.ravel(), 256, [0, 256], color='steelblue')
axes[0, 1].set_title('Orijinal Histogram')

axes[1, 0].imshow(equalized, cmap='gray')
axes[1, 0].set_title('Histogram Eşitlemesi Sonrası')
axes[1, 0].axis('off')

axes[1, 1].hist(equalized.ravel(), 256, [0, 256], color='coral')
axes[1, 1].set_title('Eşitlenmiş Histogram')

plt.suptitle('Histogram Eşitleme', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Morfolojik ve Geometrik İşlemler

Bu işlemler, nesnelerin piksel sınırlarını manipüle etmek ve görüntüyü sinir ağı giriş boyutlarına uyarlamak için kullanılır.

In [ ]:
# Morfolojik işlemler için yapısal eleman (kernel) tanımlama
kernel = np.ones((5, 5), np.uint8)

# Erozyon: Nesne sınırlarını küçültür, ince gürültüleri temizler
erosion = cv2.erode(thresh_otsu, kernel, iterations=1)

# Genişletme (Dilation): Nesne sınırlarını büyütür, boşlukları doldurur
dilation = cv2.dilate(thresh_otsu, kernel, iterations=1)

# Açma (Opening): Erozyon + Genişletme -> İnce çıkıntıları temizler
opening = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel)

# Kapama (Closing): Genişletme + Erozyon -> Nesne içi boşlukları doldurur
closing = cv2.morphologyEx(thresh_otsu, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = ['Orijinal', 'Erozyon', 'Genişletme', 
          'Açma (Opening)', 'Kapama (Closing)', 'Erozyon + Genişletme']
images = [thresh_otsu, erosion, dilation, opening, closing, 
          cv2.dilate(cv2.erode(thresh_otsu, kernel), kernel)]

for ax, img, title in zip(axes.flat, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Morfolojik İşlemler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Geometrik İşlemler

# Boyutlandırma (Resize)
# Sinir ağı giriş katmanı boyutlarına sabitleme
resized_224 = cv2.resize(img_noisy, (224, 224))  # VGG/ResNet giriş boyutu
resized_160 = cv2.resize(img_noisy, (160, 160))  # MobileNetV2 giriş boyutu

print(f"Orijinal boyut: {img_noisy.shape}")
print(f"224x224 boyut: {resized_224.shape}")
print(f"160x160 boyut: {resized_160.shape}")

# Kırpma (Cropping)
cropped = img_noisy[50:250, 100:300]  # [y1:y2, x1:x2]
print(f"Kırpılmış boyut: {cropped.shape}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(resized_224, cv2.COLOR_BGR2RGB))
axes[0].set_title('224x224 (VGG/ResNet)')

axes[1].imshow(cv2.cvtColor(resized_160, cv2.COLOR_BGR2RGB))
axes[1].set_title('160x160 (MobileNetV2)')

axes[2].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
axes[2].set_title('Kırpılmış Görsel')

for ax in axes:
    ax.axis('off')

plt.suptitle('Boyutlandırma ve Kırpma', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Görüntü İşleme Kullanım Alanları

| Kullanım Alanı | Uygulanan Teknikler |
|---|---|
| **Plaka Tanıma** | Siyah-beyaz dönüşüm, eşikleme, kenar tespiti |
| **Belge Tarama (OCR)** | Perspektif düzeltme, ikili eşikleme |
| **Nesne Takibi** | HSV renk uzayına dönüşüm, renk aralığı filtreleme |
| **Derin Öğrenme Ön İşleme** | Boyutlandırma, normalizasyon |

---
## 5. Transfer Learning (Aktarım Öğrenimi)

### 5.1 Transfer Learning Nedir?

Derin öğrenme ağlarını rastgele ağırlıklarla sıfırdan eğitmek yerine, önceden optimize edilmiş model parametrelerini yeni problemlerde kullanma yaklaşımıdır.

### 5.2 Neden Kullanılır?

- Orijinal eğitim süreçleri **milyonlarca etiketli görsel** ve **yoğun GPU süreleri** gerektirir
- Akademik ve gerçek dünya senaryolarında bu kadar büyük veri genellikle sağlanamaz
- Optimize edilmiş ağırlıkların aktarımı, **kısıtlı veri setlerinde doğruluk oranını artırır**
- **Eğitim süresini kısaltır** ve **aşırı öğrenme (overfitting) riskini azaltır**

### 5.3 İşlem Akışı

1. Temel görsel özellikleri barındıran CNN mimarisi yüklenir
2. Ağın özellik çıkaran erken katmanları **dondurulur** (gradyan güncellemeleri kapatılır)
3. Modelin son katmanları yeni etiket formatına göre yapılandırılarak **eğitime açılır**

### 5.4 Transfer Learning Stratejileri

| Strateji | Açıklama | Ne Zaman Kullanılır? |
|---|---|---|
| **1. Sabit Özellik Çıkarıcı** | Tüm katmanlar dondurulur, sadece son katman eğitilir | Düşük veri miktarı |
| **2. İnce Ayar (Fine Tuning)** | Son bloklar eğitime açılır, düşük LR kullanılır | Orta-yüksek veri miktarı |
| **3. Tam Model Eğitimi** | Tüm katmanlar eğitime açılır, pre-trained ağırlıklar başlangıç noktası | Yüksek veri + güçlü GPU |
| **4. Kısmi Katman Dondurma** | Belirli bloklar seçilerek dondurulur/açılır | Dengeli yaklaşım |
| **5. Alan Uyarlama (Domain Adaptation)** | Farklı veri dağılımlarına uyarlama | Kaynak ≠ Hedef domain |

---
## 6. Önceden Eğitilmiş Modeller ve ImageNet

### ImageNet Veri Kümesi
- **14 milyon** etiketli görsel
- **21.000+** kategori
- **1000 sınıflı** alt küme standart eğitimlerde referans olarak kabul edilir

### Popüler Mimariler Karşılaştırması

| Model | Boyut (MB) | Top-1 Acc. | Parametre | Derinlik | GPU ms |
|---|---|---|---|---|---|
| MobileNet | 16 | 70.4% | 4.3M | 55 | 3.4 |
| MobileNetV2 | 14 | 71.3% | 3.5M | 105 | 3.8 |
| ResNet50 | 98 | 74.9% | 25.6M | 107 | 4.6 |
| VGG16 | 528 | 71.3% | 138.4M | 16 | 4.2 |
| EfficientNetB0 | 29 | 77.1% | 5.3M | 132 | 4.9 |
| EfficientNetB7 | 256 | 84.3% | 66.7M | 438 | 61.6 |
| DenseNet121 | 33 | 75.0% | 8.1M | 242 | 5.4 |

### Model Seçim Kriterleri

- **Mobil ve Gömülü Sistemler:** MobileNet, MobileNetV2, EfficientNetB0
- **Yüksek Doğruluk İhtiyacı:** EfficientNetB7, ResNet152V2, ConvNeXtLarge
- **Hız Odaklı Çıkarım:** ResNet50V2, EfficientNetB1
- **Düşük Parametre Maliyeti:** DenseNet121, MobileNet

---
## 7. Transfer Learning Uygulaması - Çiçek Sınıflandırma

Bu bölümde MobileNetV2 kullanarak 5 farklı çiçek sınıfını (daisy, dandelion, roses, sunflowers, tulips) sınıflandıracağız.

### 7.1 Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pathlib
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow sürümü: {tf.__version__}")
print(f"GPU kullanılabilir mi: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 7.2 Veri Kümesinin İndirilmesi ve Hazırlanması

In [ ]:
# TensorFlow'un sağladığı çiçek veri setini indirme
dataset_url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

# Bazı sistemlerde iç içe klasör oluşabilir
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'

# Sınıf isimlerini listele
class_names = sorted([item.name for item in data_dir.glob("*") if item.is_dir()])
print("Sınıflar:", class_names)

# Her sınıftaki görsel sayısını göster
for cls in class_names:
    count = len(list((data_dir / cls).glob("*.jpg")))
    print(f"  {cls}: {count} görsel")

In [ ]:
# Veri seti parametreleri
batch_size = 32
img_size = (160, 160)  # MobileNetV2 için

# Eğitim seti (%80)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

# Doğrulama seti (%20)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

print(f"\nSınıf isimleri: {train_ds.class_names}")

### 7.3 Veri Setinden Örnek Görsellerin Görselleştirilmesi

In [ ]:
# Her sınıftan bir örnek görsel göster
plt.figure(figsize=(15, 5))
for i, class_name in enumerate(class_names):
    image_path = list((data_dir / class_name).glob("*.jpg"))[0]
    img = plt.imread(image_path)
    plt.subplot(1, len(class_names), i + 1)
    plt.imshow(img)
    plt.title(class_name)
    plt.axis("off")

plt.suptitle('Veri Setinden Örnek Görseller', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bir batch'ten örnek görseller
plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.suptitle('Eğitim Setinden Rastgele Örnekler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4 MobileNetV2 Taban Modelinin Yüklenmesi

**`include_top=False`** parametresi, orijinal 1000-sınıflı ImageNet sınıflandırma katmanını silerek sadece özellik çıkaran evrişim bloklarını alır.

**`weights='imagenet'`** parametresi, ImageNet üzerinde önceden eğitilmiş ağırlıkları yükler.

In [ ]:
# MobileNetV2 taban modelini ImageNet ağırlıklarıyla yükle
base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,       # Son sınıflandırma katmanını dahil etme
    weights='imagenet'       # ImageNet üzerinde eğitilmiş ağırlıkları kullan
)

print(f"Taban model katman sayısı: {len(base_model.layers)}")
print(f"Taban model çıktı shape: {base_model.output_shape}")

### 7.5 Önceden Eğitilmiş Katmanların Dondurulması

İlk eğitim aşamasında taban modelin ağırlıkları dondurulur:
- İlgili katmanlar **güncellenmez** → hesaplama yükü azalır
- Sınırlı veri setlerinde **overfitting riski düşer**
- Önceden öğrenilmiş genel özellikler **korunur**

In [ ]:
# Tüm taban model katmanlarını dondur
base_model.trainable = False

print(f"Dondurulan katman sayısı: {len(base_model.layers)}")
print(f"Eğitilebilir parametre sayısı: {sum(p.numpy().size for p in base_model.trainable_weights)}")
print(f"Toplam parametre sayısı: {sum(p.numpy().size for p in base_model.weights)}")

### 7.6 Sınıflandırma Katmanlarının Eklenmesi

Taban model çıktıları üzerine yeni katmanlar eklenir:

- **GlobalAveragePooling2D:** Uzamsal matris çıktısını (5x5x1280) → tek boyutlu vektöre (1280,) indirger
- **Dense(128, relu):** Özellik haritası üzerinde yeni ilişkiler öğrenir
- **Dropout(0.2):** Aşırı öğrenmeyi önlemek için rastgele nöronları kapatır
- **Dense(5, softmax):** 5 çiçek sınıfı için olasılık dağılımı üretir

In [ ]:
# Yeni sınıflandırma katmanlarını ekle
x = base_model.output
x = GlobalAveragePooling2D()(x)    # (batch, 5, 5, 1280) -> (batch, 1280)
x = Dense(128, activation='relu')(x)  # Ara katman
x = Dropout(0.2)(x)                  # %20 dropout
predictions = Dense(5, activation='softmax')(x)  # 5 sınıf

# Final modeli oluştur
model = Model(inputs=base_model.input, outputs=predictions)

# Model özeti
model.summary()

### 7.7 Özellik Çıkarımı (Feature Extraction) Eğitimi

İlk aşamada taban katmanlar dondurulur ve sadece yeni eklenen tam bağlı katman optimize edilir.

- **Optimizer:** Adam (varsayılan learning rate = 0.001)
- **Loss:** sparse_categorical_crossentropy (integer etiketler için)
- **Metric:** accuracy

In [ ]:
# Modeli derle
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Özellik çıkarımı eğitimi (5 epoch)
print("=" * 50)
print("AŞAMA 1: Özellik Çıkarımı (Feature Extraction)")
print("=" * 50)

history_fe = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

### 7.8 İnce Ayar (Fine Tuning) İşlemi

Özellik çıkarımı aşamasından sonra, modelin belirli katmanları eğitime açılır.

**Kritik Noktalar:**
- İlk 100 katman **dondurulmuş** kalır (genel özellikler korunur)
- Son katmanlar **eğitime açılır** (hedef veriye uyum sağlar)
- **Düşük öğrenme oranı (1e-5)** kullanılır → önceden öğrenilmiş ağırlıkların bozulması önlenir

In [ ]:
# Taban modeli eğitime aç
base_model.trainable = True

# İlk 100 katmanı dondur, geri kalanları eğitime aç
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Eğitilebilir ve dondurulmuş katman sayılarını göster
trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
frozen_count = sum(1 for layer in base_model.layers if not layer.trainable)
print(f"Eğitilebilir katman sayısı: {trainable_count}")
print(f"Dondurulmuş katman sayısı: {frozen_count}")

In [ ]:
# Düşük öğrenme oranı ile yeniden derle
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Çok düşük LR!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# İnce ayar eğitimi
print("=" * 50)
print("AŞAMA 2: İnce Ayar (Fine Tuning)")
print("=" * 50)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

### 7.9 Erken Durdurma ve Model Kaydetme (Opsiyonel - Uzun Eğitim İçin)

Doğrulama kayıp değeri iyileşme göstermediğinde eğitimi otomatik olarak sonlandırmak ve en iyi ağırlıkları kaydetmek için callback'ler kullanılır.

In [ ]:
# Callback'lerin tanımlanması (uzun eğitimler için önerilir)
callbacks = [
    EarlyStopping(
        monitor='val_loss',        # Doğrulama kaybını izle
        patience=3,                # 3 epoch boyunca iyileşme olmazsa dur
        restore_best_weights=True   # En iyi ağırlıkları geri yükle
    ),
    ModelCheckpoint(
        'best_model.keras',         # Kayıt dosyası
        monitor='val_loss',         # Doğrulama kaybını izle
        save_best_only=True         # Sadece en iyi modeli kaydet
    )
]

# Uzun eğitim (isteğe bağlı - yorum satırından çıkararak çalıştırabilirsiniz)
# history_long = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=20,
#     callbacks=callbacks
# )

print("Callback'ler tanımlandı. Uzun eğitim için yorum satırlarını kaldırın.")

---
## 8. Eğitim Sonuçlarının Görselleştirilmesi

In [ ]:
# Özellik çıkarımı ve ince ayar eğitim metriklerini birleştir
acc = history_fe.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_fe.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history_fe.history['loss'] + history_ft.history['loss']
val_loss = history_fe.history['val_loss'] + history_ft.history['val_loss']

# Doğruluk grafiği
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
axes[0].plot(acc, label='Eğitim Doğruluğu', linewidth=2)
axes[0].plot(val_acc, label='Doğrulama Doğruluğu', linewidth=2)
axes[0].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='İnce Ayar Başlangıcı')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Doğruluk')
axes[0].set_title('Eğitim ve Doğrulama Doğruluğu')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(loss, label='Eğitim Kaybı', linewidth=2)
axes[1].plot(val_loss, label='Doğrulama Kaybı', linewidth=2)
axes[1].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='İnce Ayar Başlangıcı')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kayıp')
axes[1].set_title('Eğitim ve Doğrulama Kaybı')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Eğitim Metrikleri (Feature Extraction → Fine Tuning)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Model Değerlendirme

### 9.1 Modelin Kaydedilmesi ve Değerlendirilmesi

In [ ]:
# Modeli kaydet
model.save('mobilenetv2_transfer.h5')
print("Model 'mobilenetv2_transfer.h5' olarak kaydedildi.")

# Doğrulama seti üzerinde değerlendir
loss, acc = model.evaluate(val_ds)
print(f"\nDoğrulama Kaybı: {loss:.4f}")
print(f"Doğrulama Doğruluğu: {acc:.2%}")

### 9.2 Karmaşıklık Matrisi (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Doğrulama setindeki tüm görselleri ve etiketleri topla
val_images = []
val_labels = []
for images, labels in val_ds:
    val_images.append(images.numpy())
    val_labels.append(labels.numpy())

val_images = np.vstack(val_images)
val_labels = np.concatenate(val_labels)

# Tahminleri al
predictions = model.predict(val_images)
predicted_classes = np.argmax(predictions, axis=1)

print(f"Toplam doğrulama görseli: {len(val_labels)}")
print(f"Doğru tahmin: {np.sum(predicted_classes == val_labels)}")
print(f"Yanlış tahmin: {np.sum(predicted_classes != val_labels)}")

In [ ]:
# Karmaşıklık matrisi oluştur ve görselleştir
cm = confusion_matrix(val_labels, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Karmaşıklık Matrisi Analizi')
plt.tight_layout()
plt.show()

# Sınıflandırma raporu
print("\nSınıflandırma Raporu:")
print("=" * 60)
print(classification_report(val_labels, predicted_classes, 
                            target_names=class_names))

### 9.3 Çıkarım (Inference) - Tek Görsel ile Tahmin

Eğitim tamamlandıktan sonra model sadece ileri yayılım (forward pass) yapar. Ağırlıklar sabittir ve güncellenmez.

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Modeli yükle (farklı bir oturumda çalışıyorsanız)
# model = load_model('mobilenetv2_transfer.h5')

def predict_flower(img_path, model, class_names):
    """
    Tek bir görsel üzerinde çıkarım (inference) yapan fonksiyon.
    
    Args:
        img_path: Görsel dosya yolu
        model: Eğitilmiş Keras modeli
        class_names: Sınıf isimleri listesi
    
    Returns:
        Tahmin edilen sınıf ve olasılık
    """
    # Görseli yükle ve ön işle
    img = image.load_img(img_path, target_size=(160, 160))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Batch boyutu ekle
    
    # Tahmin yap
    pred = model.predict(img_array, verbose=0)
    class_idx = np.argmax(pred)
    confidence = pred[0][class_idx]
    
    return class_names[class_idx], confidence, pred[0]

# Doğrulama setinden rastgele bir görsel ile test
test_class = class_names[0]  # İlk sınıf
test_images = list((data_dir / test_class).glob("*.jpg"))

if test_images:
    test_img_path = str(test_images[5])  # 6. görsel
    predicted_class, confidence, all_probs = predict_flower(
        test_img_path, model, class_names
    )
    
    # Görseli ve tahmini göster
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    img = plt.imread(test_img_path)
    axes[0].imshow(img)
    axes[0].set_title(f'Gerçek: {test_class}\nTahmin: {predicted_class} ({confidence:.1%})')
    axes[0].axis('off')
    
    # Olasılık dağılımı
    colors = ['#2ecc71' if c == predicted_class else '#3498db' for c in class_names]
    axes[1].barh(class_names, all_probs, color=colors)
    axes[1].set_xlabel('Olasılık')
    axes[1].set_title('Sınıf Olasılıkları')
    axes[1].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.show()

---
## 10. Açıklanabilir Yapay Zeka - Grad-CAM

Grad-CAM (Gradient-weighted Class Activation Mapping), modelin sınıflandırma kararını hangi bölgelere odaklanarak verdiğini görselleştirir.

**Çalışma prensibi:** Son evrişim katmanındaki gradyan akışı hesaplanarak piksel bazlı ısı haritası üretilir.

- **Sıcak renkler (kırmızı/sarı):** Modelin en çok dikkat ettiği bölgeler
- **Soğuk renkler (mavi/lacivert):** Model tarafından göz ardı edilen arka plan

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Grad-CAM ısı haritası üretir.
    
    Temel mantık:
    1. Son evrişim katmanının çıktısına ve modelin son çıktısına erişen 
       bir gradyan modeli oluştur
    2. Hedef sınıf için gradyanları hesapla
    3. Gradyanların ortalamasını alarak her bir filtrenin önem ağırlığını bul
    4. Önem ağırlıkları ile özellik haritalarını çarparak ısı haritası oluştur
    """
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    # Gradyanları hesapla
    grads = tape.gradient(class_channel, last_conv_layer_output)
    
    # Her filtrenin önem ağırlığı (global average pooling)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Ağırlıklı toplam ile ısı haritası
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize et
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


def display_gradcam(img_path, model, last_conv_layer_name, class_names):
    """Grad-CAM sonucunu orijinal görsel üzerine bindirerek gösterir."""
    # Görseli hazırla
    img = image.load_img(img_path, target_size=(160, 160))
    img_array = image.img_to_array(img)
    img_array_batch = np.expand_dims(img_array, axis=0)
    
    # Tahmin yap
    pred = model.predict(img_array_batch, verbose=0)
    pred_class = class_names[np.argmax(pred)]
    
    # Grad-CAM hesapla
    heatmap = make_gradcam_heatmap(img_array_batch, model, last_conv_layer_name)
    
    # Isı haritasını görsel boyutuna yeniden boyutlandır
    heatmap_resized = cv2.resize(heatmap, (160, 160))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # Orijinal görsel üzerine bindir
    superimposed = (heatmap_colored * 0.4 + img_array * 0.6).astype(np.uint8)
    
    # Görselleştir
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img_array.astype(np.uint8))
    axes[0].set_title(f'Orijinal Görsel\nTahmin: {pred_class}')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Grad-CAM Isı Haritası')
    axes[1].axis('off')
    
    axes[2].imshow(superimposed)
    axes[2].set_title('Grad-CAM Odak Analizi')
    axes[2].axis('off')
    
    plt.suptitle('Açıklanabilir Yapay Zeka - Grad-CAM', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Son evrişim katmanının adını bul
# MobileNetV2'nin son evrişim katmanı genellikle 'out_relu' veya 'Conv_1' olur
last_conv_name = None
for layer in reversed(base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_name = layer.name
        break

print(f"Son evrişim katmanı: {last_conv_name}")

# Grad-CAM görselleştirme
if test_images and last_conv_name:
    display_gradcam(str(test_images[5]), model, last_conv_name, class_names)

---
## 11. Video İşleme ve Gerçek Zamanlı Çıkarım

### 11.1 OpenCV ile Video Verisi Okuma

Video verisi ardışık görüntü karelerinden oluşur. Her kare standart matris formatına dönüştürülerek işlenir.

> **Not:** Aşağıdaki kodlar kamera erişimi gerektirir. Jupyter Notebook'ta doğrudan çalışmayabilir; `.py` dosyası olarak çalıştırmanız önerilir.

In [ ]:
# OpenCV ile kamera görüntüsü okuma
# NOT: Bu kod kamera erişimi gerektirir. 
# Jupyter'da çalışmayabilir - .py dosyası olarak çalıştırın.

'''
import cv2

cap = cv2.VideoCapture(0)  # 0 = varsayılan kamera

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    cv2.imshow('Video Ekrani', frame)
    
    # 'q' tuşuna basılırsa çık
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''
print("Video okuma kodu hazır. Kamera erişimi olan ortamda çalıştırın.")

### 11.2 Video Kareleri Üzerinde Gerçek Zamanlı Model Çıkarımı

In [ ]:
# Gerçek zamanlı çiçek sınıflandırma (kamera ile)
# NOT: Kamera erişimi gerektirir.

'''
import cv2
import numpy as np

# Eğitilmiş modeli yükle
# model = load_model('mobilenetv2_transfer.h5')
class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Kareyi model giriş boyutuna yeniden boyutlandır
    img_resized = cv2.resize(frame, (160, 160))
    img_array = np.expand_dims(img_resized, axis=0)
    
    # Tahmin yap
    pred = model.predict(img_array, verbose=0)
    class_idx = np.argmax(pred)
    confidence = pred[0][class_idx]
    label = f"{class_names[class_idx]}: {confidence:.1%}"
    
    # Sonucu kare üzerine yaz
    cv2.putText(frame, label, (50, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow('Cikarim Ekrani', frame)
    
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''
print("Gerçek zamanlı çıkarım kodu hazır. Kamera erişimi olan ortamda çalıştırın.")

---
## 12. CNN Mimarileri Özet Karşılaştırması

### VGG-16
- **Yıl:** 2014
- **Yapı:** 13 evrişim + 3 tam bağlı = 16 öğrenilebilir katman
- **Özellik:** Ardışık 3x3 filtrelerle derinlik sağlar
- **Parametre:** ~138M
- **Kullanım:** Transfer learning için özellik çıkarıcı olarak yaygın

### ResNet50
- **Yıl:** 2015
- **Yapı:** 49 evrişim + 1 tam bağlı katman, 5 ana blok aşaması
- **Özellik:** Kalıntı bağlantıları (residual connections) ile gradyan kaybını önler
- **Parametre:** ~25.6M
- **Kullanım:** ImageNet referans modeli

### EfficientNet
- **Yıl:** 2019
- **Yapı:** B0-B7 arası 8 farklı ölçek
- **Özellik:** Derinlik, genişlik ve çözünürlüğü orantılı olarak ölçeklendirir
- **Parametre:** 5.3M (B0) → 66.7M (B7)
- **Kullanım:** Verimlilik/doğruluk dengesi

### MobileNetV2
- **Yıl:** 2018
- **Yapı:** Inverted residual blocks, depthwise separable convolution
- **Özellik:** Mobil/gömülü sistemler için optimize edilmiş
- **Parametre:** ~3.5M
- **Kullanım:** Edge deployment, gerçek zamanlı uygulamalar

### DenseNet
- **Yıl:** 2017
- **Yapı:** Her katman önceki tüm katmanların çıktılarını alır
- **Özellik:** Bilgi kaybını önler, gradyan akışını güçlendirir
- **Parametre:** 8.1M (121) → 20.2M (201)
- **Kullanım:** Parametre-verimli yüksek doğruluk

### ConvNeXt
- **Yıl:** 2022 (Meta AI)
- **Yapı:** ResNet tabanlı, modern güncellemelerle (LayerNorm, GELU, büyük kernel)
- **Özellik:** CNN'leri Vision Transformer seviyesine ulaştırma hedefi
- **Kullanım:** Yüksek performanslı modern CNN

### Vision Transformer (ViT) & Swin Transformer
- **ViT:** Görüntüyü patch'lere ayırıp self-attention uygular
- **Swin:** Hiyerarşik pencereler ile yerel+küresel özellik öğrenimi
- Büyük veri setlerinde CNN'lerden daha yüksek doğruluk sağlar

---
## 13. Bonus: Farklı Taban Modellerle Deneme

Keras Applications aracılığıyla farklı mimarileri kolayca deneyebilirsiniz.

In [ ]:
# Farklı taban modelleri yükleme örnekleri

from tensorflow.keras.applications import (
    ResNet50, VGG16, EfficientNetB0, DenseNet121
)

# ResNet50
# resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# VGG16
# vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# EfficientNetB0
# effnet_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# DenseNet121
# densenet_base = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

print("Keras Applications desteklediği bazı mimariler:")
print("- MobileNetV2: Mobil/gömülü sistemler için")
print("- ResNet50: Residual bloklar ile derin ağlar")
print("- VGG16: Klasik 3x3 filtre yaklaşımı")
print("- EfficientNetB0-B7: Verimli ölçeklendirme")
print("- DenseNet121: Yoğun bağlantılar")
print("\nHer birini 'include_top=False' ile yükleyip transfer learning uygulayabilirsiniz.")

---
## Özet

Bu atölyede öğrendiklerimiz:

1. **Görüntü İşleme** ile ham pikselleri derin öğrenme modellerine hazır hale getirdik
2. **Transfer Learning** kavramını ve neden kullanıldığını anladık
3. **Feature Extraction** ve **Fine Tuning** stratejilerini uyguladık
4. **MobileNetV2** ile çiçek sınıflandırma modeli geliştirdik
5. **Confusion Matrix** ve **Grad-CAM** ile model performansını değerlendirdik
6. **VGG, ResNet, EfficientNet, DenseNet, ConvNeXt, ViT** mimarilerini karşılaştırdık
7. **Video işleme** ve gerçek zamanlı çıkarım temellerini gördük

---

**Kaynaklar:**
- [Keras Applications](https://keras.io/api/applications/)
- [TensorFlow Transfer Learning Tutorial](https://www.tensorflow.org/tutorials/images/transfer_learning)
- [OpenCV Documentation](https://docs.opencv.org/)
- [ImageNet](https://www.image-net.org/)